In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")
dbutils.widgets.text("null_high_severity_pct", "10")
dbutils.widgets.text("dq_sample_limit", "20")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
null_high_severity_pct = float(dbutils.widgets.get("null_high_severity_pct"))
dq_sample_limit = int(dbutils.widgets.get("dq_sample_limit"))

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

## Step 6 - Gold / Reporting Layer

| Gold Table | Joins | Purpose |
|---|---|---|
| `gold_claims_summary` | claims + premiums | Frequency, severity & settlement by peril, status, region |
| `gold_loss_ratio_by_risk` | premiums + claims | Loss ratio by insurer, risk band, building type, mitigation |
| `gold_event_loss_summary` | claims + events + premiums | Cat vs non-cat losses per named cyclone event |
| `gold_portfolio_exposure` | premiums + risk zone | Sum insured, policy count & premium rate by risk segment |
| `gold_claims_development` | claims + premiums | Reporting lag, IBNR indicators & reserve by peril & month |

In [ ]:
from actuarial_claim_pipeline.gold import build_gold_tables

gold_tables = build_gold_tables(spark, catalog, schema)

for name, df in gold_tables.items():
    print(f"{catalog}.{schema}.{name}: {df.count():,} rows")

In [ ]:
display(spark.table(f"{catalog}.{schema}.gold_loss_ratio_by_risk").limit(20))

In [ ]:
display(spark.table(f"{catalog}.{schema}.gold_event_loss_summary").limit(20))

In [ ]:
display(spark.table(f"{catalog}.{schema}.gold_portfolio_exposure").limit(20))

In [ ]:
display(spark.table(f"{catalog}.{schema}.gold_claims_development").limit(20))